In [2]:
# ============================================================
# 상위 3개 모델 결합 방식 개선
#
# 사용하는 모델:
#   - Gemma4-31B
#   - Qwen3.8-27B
#   - Qwen3.5-27B
#
# 입력 폴더:
#   MyDrive/ensemble/
#
# 출력 폴더:
#   MyDrive/ensemble/top3_advanced_outputs/
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Google Drive 안전 마운트
# ============================================================

DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive")

if DEFAULT_DRIVE_ROOT.is_dir():
    DRIVE_ROOT = DEFAULT_DRIVE_ROOT
    print(f"기존 Google Drive 사용: {DRIVE_ROOT}")

else:
    from google.colab import drive

    mount_index = 0

    while True:
        if mount_index == 0:
            mount_point = Path("/content/gdrive_mount")
        else:
            mount_point = Path(
                f"/content/gdrive_mount_{mount_index}"
            )

        if not mount_point.exists():
            mount_point.mkdir(parents=True)
            break

        if mount_point.is_dir() and not any(mount_point.iterdir()):
            break

        mount_index += 1

    drive.mount(
        str(mount_point),
        force_remount=False,
    )

    DRIVE_ROOT = mount_point / "MyDrive"


# ============================================================
# 2. 경로 설정
# ============================================================

ENSEMBLE_ROOT = DRIVE_ROOT / "Ensemble"

OUTPUT_ROOT = (
    ENSEMBLE_ROOT
    / "top3_advanced_outputs"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_FILES = {
    "gemma4_31b": (
        ENSEMBLE_ROOT
        / "gemma_31b_test_probabilities_03_final_other.csv"
    ),

    "qwen38_27b": (
        ENSEMBLE_ROOT
        / "qwen38_27b_test_probabilities_03_final_other.csv"
    ),

    "qwen35_27b": (
        ENSEMBLE_ROOT
        / "qwen35_27b_test_probabilities_03_final_other.csv"
    ),
}

LABELS = np.array(["a", "b", "c", "d"])
PROBABILITY_COLUMNS = ["p_a", "p_b", "p_c", "p_d"]
EXPECTED_ROW_COUNT = 6714

print(f"입력 폴더: {ENSEMBLE_ROOT}")
print(f"출력 폴더: {OUTPUT_ROOT}")


# ============================================================
# 3. 입력 파일 확인
# ============================================================

missing_files = {
    model_name: path
    for model_name, path in MODEL_FILES.items()
    if not path.is_file()
}

if missing_files:
    message = "\n".join(
        f"- {model_name}: {path}"
        for model_name, path in missing_files.items()
    )

    raise FileNotFoundError(
        "\n다음 파일을 찾을 수 없습니다.\n"
        f"{message}"
    )

print()
print("=" * 80)
print("입력 파일")
print("=" * 80)

for model_name, path in MODEL_FILES.items():
    print(f"[FOUND] {model_name}")
    print(f"        {path}")


# ============================================================
# 4. 확률 CSV 로딩 및 검증
# ============================================================

PROBABILITY_COLUMN_SCHEMES = [
    ["p_a", "p_b", "p_c", "p_d"],
    ["a", "b", "c", "d"],
    ["prob_a", "prob_b", "prob_c", "prob_d"],
    ["score_a", "score_b", "score_c", "score_d"],
]


def load_probability_csv(model_name, path):
    frame = pd.read_csv(path)

    frame.columns = [
        str(column).strip()
        for column in frame.columns
    ]

    if "id" not in frame.columns:
        raise ValueError(
            f"[{model_name}] id 열이 없습니다."
        )

    detected_columns = None

    for scheme in PROBABILITY_COLUMN_SCHEMES:
        if all(column in frame.columns for column in scheme):
            detected_columns = scheme
            break

    if detected_columns is None:
        raise ValueError(
            f"[{model_name}] 확률 열이 없습니다.\n"
            f"현재 열: {frame.columns.tolist()}"
        )

    frame = frame.rename(
        columns=dict(
            zip(
                detected_columns,
                PROBABILITY_COLUMNS,
            )
        )
    )

    frame["id"] = (
        frame["id"]
        .astype(str)
        .str.strip()
    )

    if frame["id"].duplicated().any():
        duplicate_ids = frame.loc[
            frame["id"].duplicated(keep=False),
            "id",
        ].unique()

        raise ValueError(
            f"[{model_name}] 중복 ID가 있습니다.\n"
            f"예시: {duplicate_ids[:10].tolist()}"
        )

    if len(frame) != EXPECTED_ROW_COUNT:
        raise ValueError(
            f"[{model_name}] 행 수 오류\n"
            f"예상: {EXPECTED_ROW_COUNT:,}\n"
            f"실제: {len(frame):,}"
        )

    for column in PROBABILITY_COLUMNS:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    probabilities = frame[
        PROBABILITY_COLUMNS
    ].to_numpy(dtype=np.float64)

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"[{model_name}] NaN 또는 무한대 확률이 있습니다."
        )

    if (probabilities < -1e-8).any():
        raise ValueError(
            f"[{model_name}] 음수 확률이 있습니다."
        )

    probabilities = np.clip(
        probabilities,
        0.0,
        None,
    )

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True,
    )

    if (row_sums <= 0).any():
        raise ValueError(
            f"[{model_name}] 확률 합이 0인 행이 있습니다."
        )

    probabilities /= row_sums

    result = pd.DataFrame({
        "id": frame["id"].to_numpy(),
        "p_a": probabilities[:, 0],
        "p_b": probabilities[:, 1],
        "p_c": probabilities[:, 2],
        "p_d": probabilities[:, 3],
    })

    print(
        f"[LOADED] {model_name:<12} "
        f"rows={len(result):,}, "
        f"columns={detected_columns}"
    )

    return result


# ============================================================
# 5. 모델 파일 로딩
# ============================================================

print()
print("=" * 80)
print("확률 CSV 검증")
print("=" * 80)

model_frames = {
    model_name: load_probability_csv(
        model_name,
        path,
    )
    for model_name, path in MODEL_FILES.items()
}


# ============================================================
# 6. ID 정렬
# ============================================================

REFERENCE_MODEL = "gemma4_31b"

reference_ids = model_frames[
    REFERENCE_MODEL
]["id"].tolist()

reference_id_set = set(reference_ids)
row_count = len(reference_ids)

probabilities = {}

for model_name, frame in model_frames.items():
    current_id_set = set(frame["id"])

    missing_ids = (
        reference_id_set
        - current_id_set
    )

    extra_ids = (
        current_id_set
        - reference_id_set
    )

    if missing_ids or extra_ids:
        raise ValueError(
            f"[{model_name}] ID 구성이 다릅니다.\n"
            f"누락: {len(missing_ids):,}\n"
            f"추가: {len(extra_ids):,}"
        )

    aligned = (
        frame
        .set_index("id")
        .loc[reference_ids]
    )

    probabilities[model_name] = (
        aligned[PROBABILITY_COLUMNS]
        .to_numpy(dtype=np.float64)
    )

print(f"ID 정렬 완료: {row_count:,}개")


# ============================================================
# 7. 공통 함수
# ============================================================

def normalize_probabilities(values):
    values = np.clip(
        values,
        1e-12,
        None,
    )

    return (
        values
        / values.sum(axis=1, keepdims=True)
    )


def predict_from_probabilities(values):
    return LABELS[
        np.argmax(values, axis=1)
    ]


def calculate_margin(values):
    sorted_values = np.sort(
        values,
        axis=1,
    )

    return (
        sorted_values[:, -1]
        - sorted_values[:, -2]
    )


def apply_temperature(values, temperature):
    """
    temperature > 1:
        확률을 부드럽게 만들어 과신을 줄임

    temperature < 1:
        확률을 더 날카롭게 만듦
    """
    values = np.clip(
        values,
        1e-12,
        1.0,
    )

    adjusted = np.power(
        values,
        1.0 / temperature,
    )

    return normalize_probabilities(adjusted)


def weighted_soft_vote(
    probability_map,
    weights,
):
    total_weight = float(
        sum(weights.values())
    )

    normalized_weights = {
        model_name: weight / total_weight
        for model_name, weight in weights.items()
    }

    combined = np.zeros(
        (row_count, 4),
        dtype=np.float64,
    )

    for model_name, weight in normalized_weights.items():
        combined += (
            weight
            * probability_map[model_name]
        )

    return normalize_probabilities(combined)


def save_submission(
    method_name,
    answers,
):
    submission = pd.DataFrame({
        "id": reference_ids,
        "answer": answers,
    })

    if len(submission) != EXPECTED_ROW_COUNT:
        raise RuntimeError(
            f"[{method_name}] 행 수가 잘못되었습니다."
        )

    if submission["id"].duplicated().any():
        raise RuntimeError(
            f"[{method_name}] 중복 ID가 있습니다."
        )

    invalid_answers = (
        set(submission["answer"])
        - set(LABELS)
    )

    if invalid_answers:
        raise RuntimeError(
            f"[{method_name}] 잘못된 answer: "
            f"{invalid_answers}"
        )

    output_path = (
        OUTPUT_ROOT
        / f"submission_{method_name}.csv"
    )

    submission.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    return submission, output_path


# ============================================================
# 8. 개별 모델 예측과 확신도
# ============================================================

model_predictions = {
    model_name: predict_from_probabilities(values)
    for model_name, values in probabilities.items()
}

model_margins = {
    model_name: calculate_margin(values)
    for model_name, values in probabilities.items()
}

gemma_predictions = model_predictions["gemma4_31b"]
qwen38_predictions = model_predictions["qwen38_27b"]
qwen35_predictions = model_predictions["qwen35_27b"]

top2_agree = (
    gemma_predictions
    == qwen38_predictions
)

all_three_agree = (
    top2_agree
    & (gemma_predictions == qwen35_predictions)
)

all_three_different = (
    (gemma_predictions != qwen38_predictions)
    & (gemma_predictions != qwen35_predictions)
    & (qwen38_predictions != qwen35_predictions)
)

print()
print("=" * 80)
print("모델 예측 일치 현황")
print("=" * 80)

print(
    f"Gemma31B와 Qwen38 일치: "
    f"{top2_agree.sum():,}개 "
    f"({top2_agree.mean():.2%})"
)

print(
    f"세 모델 모두 일치: "
    f"{all_three_agree.sum():,}개 "
    f"({all_three_agree.mean():.2%})"
)

print(
    f"세 모델 모두 다른 답: "
    f"{all_three_different.sum():,}개 "
    f"({all_three_different.mean():.2%})"
)


# ============================================================
# 9. 방법 1: 기존 최고 Soft-voting
#
# 공개 점수: 0.96127
# ============================================================

BASELINE_WEIGHTS = {
    "qwen38_27b": 0.40,
    "gemma4_31b": 0.35,
    "qwen35_27b": 0.25,
}

baseline_probabilities = weighted_soft_vote(
    probabilities,
    BASELINE_WEIGHTS,
)

baseline_answers = predict_from_probabilities(
    baseline_probabilities
)


# ============================================================
# 10. 방법 2: 점수 조정 Soft-voting
# ============================================================

TUNED_WEIGHTS = {
    "gemma4_31b": 0.42,
    "qwen38_27b": 0.40,
    "qwen35_27b": 0.18,
}

tuned_probabilities = weighted_soft_vote(
    probabilities,
    TUNED_WEIGHTS,
)

tuned_answers = predict_from_probabilities(
    tuned_probabilities
)


# ============================================================
# 11. 방법 3: Temperature 보정 Soft-voting
#
# 상대적으로 약한 Qwen35의 과신을 더 많이 완화
# ============================================================

TEMPERATURES = {
    "gemma4_31b": 1.05,
    "qwen38_27b": 1.10,
    "qwen35_27b": 1.30,
}

temperature_probabilities = {
    model_name: apply_temperature(
        probabilities[model_name],
        TEMPERATURES[model_name],
    )
    for model_name in TUNED_WEIGHTS
}

temperature_ensemble_probabilities = weighted_soft_vote(
    temperature_probabilities,
    TUNED_WEIGHTS,
)

temperature_answers = predict_from_probabilities(
    temperature_ensemble_probabilities
)


# ============================================================
# 12. 방법 4: Top3 다수결 + Gemma 타이브레이크
#
# 두 모델이 같은 답이면 그 답을 사용
# 세 모델이 모두 다르면 Gemma31B 답을 사용
# ============================================================

majority_answers = np.empty(
    row_count,
    dtype="<U1",
)

for index in range(row_count):
    predictions_at_index = [
        gemma_predictions[index],
        qwen38_predictions[index],
        qwen35_predictions[index],
    ]

    counts = {
        label: predictions_at_index.count(label)
        for label in LABELS
    }

    majority_label = max(
        counts,
        key=counts.get,
    )

    if counts[majority_label] >= 2:
        majority_answers[index] = majority_label
    else:
        majority_answers[index] = (
            gemma_predictions[index]
        )


# ============================================================
# 13. 방법 5: Top2 합의 보호 Hybrid
#
# Gemma31B와 Qwen38이 같은 답:
#   두 모델의 합의 답을 강제로 사용
#
# 두 모델이 다른 답:
#   Temperature soft-voting 결과 사용
# ============================================================

hybrid_answers = temperature_answers.copy()

hybrid_answers[top2_agree] = (
    gemma_predictions[top2_agree]
)


# ============================================================
# 14. 방법 6: Confidence-gated 결합
#
# 1. Gemma31B와 Qwen38이 같으면 그 답
# 2. 둘이 다르면 Qwen35가 동의하는 쪽을 선택
# 3. 세 모델이 전부 다르면 Gemma31B를 기본 선택
# 4. 단, Qwen38 margin이 Gemma보다 0.10 이상 크면 Qwen38 선택
# ============================================================

CONFIDENCE_ADVANTAGE_THRESHOLD = 0.10

temperature_model_predictions = {
    model_name: predict_from_probabilities(values)
    for model_name, values in temperature_probabilities.items()
}

temperature_model_margins = {
    model_name: calculate_margin(values)
    for model_name, values in temperature_probabilities.items()
}

gated_answers = np.empty(
    row_count,
    dtype="<U1",
)

for index in range(row_count):
    gemma_answer = gemma_predictions[index]
    qwen38_answer = qwen38_predictions[index]
    qwen35_answer = qwen35_predictions[index]

    # 상위 두 모델 합의
    if gemma_answer == qwen38_answer:
        gated_answers[index] = gemma_answer
        continue

    # Qwen35가 Gemma와 합의
    if qwen35_answer == gemma_answer:
        gated_answers[index] = gemma_answer
        continue

    # Qwen35가 Qwen38과 합의
    if qwen35_answer == qwen38_answer:
        gated_answers[index] = qwen38_answer
        continue

    # 세 모델이 모두 다른 경우
    gemma_margin = temperature_model_margins[
        "gemma4_31b"
    ][index]

    qwen38_margin = temperature_model_margins[
        "qwen38_27b"
    ][index]

    if (
        qwen38_margin
        > gemma_margin + CONFIDENCE_ADVANTAGE_THRESHOLD
    ):
        gated_answers[index] = qwen38_answer
    else:
        gated_answers[index] = gemma_answer


# ============================================================
# 15. 모든 제출 파일 저장
# ============================================================

METHOD_RESULTS = {
    "baseline_top3": {
        "answers": baseline_answers,
        "probabilities": baseline_probabilities,
    },

    "tuned_soft": {
        "answers": tuned_answers,
        "probabilities": tuned_probabilities,
    },

    "temperature_soft": {
        "answers": temperature_answers,
        "probabilities": temperature_ensemble_probabilities,
    },

    "majority_gemma_tiebreak": {
        "answers": majority_answers,
        "probabilities": None,
    },

    "hybrid_top2_guard": {
        "answers": hybrid_answers,
        "probabilities": temperature_ensemble_probabilities,
    },

    "confidence_gated": {
        "answers": gated_answers,
        "probabilities": None,
    },
}

saved_submissions = {}

print()
print("=" * 80)
print("제출 파일 저장")
print("=" * 80)

for method_name, result in METHOD_RESULTS.items():
    submission, path = save_submission(
        method_name=method_name,
        answers=result["answers"],
    )

    saved_submissions[method_name] = submission

    print(f"[SAVED] {method_name}")
    print(f"        {path}")


# ============================================================
# 16. 기존 최고 제출과 차이 비교
# ============================================================

comparison = pd.DataFrame({
    "id": reference_ids,
    "pred_gemma4_31b": gemma_predictions,
    "pred_qwen38_27b": qwen38_predictions,
    "pred_qwen35_27b": qwen35_predictions,
    "top2_agree": top2_agree,
    "all_three_agree": all_three_agree,
    "all_three_different": all_three_different,
    "margin_gemma4_31b": model_margins["gemma4_31b"],
    "margin_qwen38_27b": model_margins["qwen38_27b"],
    "margin_qwen35_27b": model_margins["qwen35_27b"],
})

summary_rows = []

for method_name, result in METHOD_RESULTS.items():
    answers = result["answers"]

    changed_mask = (
        answers
        != baseline_answers
    )

    changed_when_top2_agree = (
        changed_mask
        & top2_agree
    )

    changed_when_top2_disagree = (
        changed_mask
        & ~top2_agree
    )

    comparison[
        f"answer_{method_name}"
    ] = answers

    comparison[
        f"changed_{method_name}"
    ] = changed_mask

    summary_rows.append({
        "method": method_name,
        "changed_vs_baseline": int(
            changed_mask.sum()
        ),
        "changed_ratio": float(
            changed_mask.mean()
        ),
        "changed_when_top2_agree": int(
            changed_when_top2_agree.sum()
        ),
        "changed_when_top2_disagree": int(
            changed_when_top2_disagree.sum()
        ),
    })

comparison_summary = pd.DataFrame(
    summary_rows
)

comparison_path = (
    OUTPUT_ROOT
    / "top3_method_prediction_comparison.csv"
)

summary_path = (
    OUTPUT_ROOT
    / "top3_method_comparison_summary.csv"
)

comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8",
)

comparison_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 17. Soft-voting 상세 확률 저장
# ============================================================

soft_diagnostics = pd.DataFrame({
    "id": reference_ids,

    "baseline_answer": baseline_answers,
    "baseline_p_a": baseline_probabilities[:, 0],
    "baseline_p_b": baseline_probabilities[:, 1],
    "baseline_p_c": baseline_probabilities[:, 2],
    "baseline_p_d": baseline_probabilities[:, 3],
    "baseline_margin": calculate_margin(
        baseline_probabilities
    ),

    "tuned_answer": tuned_answers,
    "tuned_p_a": tuned_probabilities[:, 0],
    "tuned_p_b": tuned_probabilities[:, 1],
    "tuned_p_c": tuned_probabilities[:, 2],
    "tuned_p_d": tuned_probabilities[:, 3],
    "tuned_margin": calculate_margin(
        tuned_probabilities
    ),

    "temperature_answer": temperature_answers,
    "temperature_p_a": (
        temperature_ensemble_probabilities[:, 0]
    ),
    "temperature_p_b": (
        temperature_ensemble_probabilities[:, 1]
    ),
    "temperature_p_c": (
        temperature_ensemble_probabilities[:, 2]
    ),
    "temperature_p_d": (
        temperature_ensemble_probabilities[:, 3]
    ),
    "temperature_margin": calculate_margin(
        temperature_ensemble_probabilities
    ),
})

soft_diagnostics_path = (
    OUTPUT_ROOT
    / "top3_soft_voting_diagnostics.csv"
)

soft_diagnostics.to_csv(
    soft_diagnostics_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# 18. 결과 출력
# ============================================================

print()
print("=" * 80)
print("상위 3개 모델 결합 완료")
print("=" * 80)

print(f"전체 샘플: {row_count:,}")
print(f"결과 폴더: {OUTPUT_ROOT}")

print()
print("기존 최고 방식 대비 변경 수:")

display(
    comparison_summary.sort_values(
        "changed_vs_baseline"
    )
)

print()
print("추천 제출 순서:")

recommended_order = [
    "hybrid_top2_guard",
    "temperature_soft",
    "confidence_gated",
    "tuned_soft",
    "majority_gemma_tiebreak",
    "baseline_top3",
]

for rank, method_name in enumerate(
    recommended_order,
    start=1,
):
    print(
        f"{rank}. "
        f"{OUTPUT_ROOT / f'submission_{method_name}.csv'}"
    )

print()
print("추가 분석 파일:")
print(comparison_path)
print(summary_path)
print(soft_diagnostics_path)

Mounted at /content/gdrive_mount_1
입력 폴더: /content/gdrive_mount_1/MyDrive/Ensemble
출력 폴더: /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs

입력 파일
[FOUND] gemma4_31b
        /content/gdrive_mount_1/MyDrive/Ensemble/gemma_31b_test_probabilities_03_final_other.csv
[FOUND] qwen38_27b
        /content/gdrive_mount_1/MyDrive/Ensemble/qwen38_27b_test_probabilities_03_final_other.csv
[FOUND] qwen35_27b
        /content/gdrive_mount_1/MyDrive/Ensemble/qwen35_27b_test_probabilities_03_final_other.csv

확률 CSV 검증
[LOADED] gemma4_31b   rows=6,714, columns=['p_a', 'p_b', 'p_c', 'p_d']
[LOADED] qwen38_27b   rows=6,714, columns=['a', 'b', 'c', 'd']
[LOADED] qwen35_27b   rows=6,714, columns=['p_a', 'p_b', 'p_c', 'p_d']
ID 정렬 완료: 6,714개

모델 예측 일치 현황
Gemma31B와 Qwen38 일치: 6,323개 (94.18%)
세 모델 모두 일치: 6,162개 (91.78%)
세 모델 모두 다른 답: 41개 (0.61%)

제출 파일 저장
[SAVED] baseline_top3
        /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_baseline_top3.csv
[SAVED] tuned_soft
  

,method,changed_vs_baseline,changed_ratio,changed_when_top2_agree,changed_when_top2_disagree
0,baseline_top3,0,0.000000,0,0
1,tuned_soft,39,0.005809,0,39
5,confidence_gated,52,0.007745,1,51
2,temperature_soft,53,0.007894,0,53
4,hybrid_top2_guard,54,0.008043,1,53
3,majority_gemma_tiebreak,60,0.008937,1,59



추천 제출 순서:
1. /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_hybrid_top2_guard.csv
2. /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_temperature_soft.csv
3. /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_confidence_gated.csv
4. /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_tuned_soft.csv
5. /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_majority_gemma_tiebreak.csv
6. /content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/submission_baseline_top3.csv

추가 분석 파일:
/content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/top3_method_prediction_comparison.csv
/content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/top3_method_comparison_summary.csv
/content/gdrive_mount_1/MyDrive/Ensemble/top3_advanced_outputs/top3_soft_voting_diagnostics.csv
